# Thesis literature map — Quantum computing in finance

This notebook explores the bibliography under `data/100_TESI/` and links each **level** to runnable scripts in `quantum/tesi/`.

> PDFs remain local (gitignored). The catalog in `papers_catalog.yaml` lists objectives and arXiv IDs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "quantum" / "tesi").exists():
    ROOT = ROOT.parent  # if launched from quantum/tesi

sys.path.insert(0, str(ROOT / "quantum"))
sys.path.insert(0, str(ROOT / "quantum" / "tesi"))

import matplotlib.pyplot as plt
import pandas as pd

from tesi._lib.catalog import load_catalog, list_local_pdfs, summarize_catalog, thesis_data_dir

## 1. Catalog overview

In [ ]:
catalog = load_catalog()
rows = []
for level in catalog["levels"]:
    for paper in level["papers"]:
        rows.append({
            "level": level["id"],
            "level_title": level["title"],
            "themes": ", ".join(level["themes"]),
            "paper": paper["title"],
            "objective": paper.get("objective", ""),
            "arxiv": paper.get("arxiv", ""),
        })

df = pd.DataFrame(rows)
df.head(10)

In [ ]:
counts = df.groupby("level").size()
counts.plot(kind="bar", title="Papers per level", color="steelblue")
plt.ylabel("count")
plt.tight_layout()

## 2. Local files in `data/100_TESI`

In [ ]:
data_dir = thesis_data_dir()
pdfs = list_local_pdfs(data_dir)
print(f"Folder: {data_dir}")
print(f"PDFs on disk: {len(pdfs)}")
if len(pdfs) == 0:
    print("Run: powershell -File data/100_TESI/download_papers.ps1")

## 3. Bank of Finland survey (2025) — key themes

Source: HTML saved under `Livello_1_Survey_Industry/`.

In [ ]:
import re

html_path = data_dir / "Livello_1_Survey_Industry" / (
    "Bank of Finland Quantum computing is coming financial sector ready 2025.html"
)
text = html_path.read_text(encoding="utf-8", errors="ignore") if html_path.exists() else ""
headings = [re.sub(r"<[^>]+>", "", h).strip() for h in re.findall(r"<h2[^>]*>(.*?)</h2>", text, re.I|re.S)]
for h in headings:
    print("•", h)

**Survey takeaways (BoF 2025):**
- Opportunities: risk management, investment activities, information security
- Risks: decryption / post-quantum migration
- ~10% of firms ran quantum trials; most expect impact within 10 years

## 4. Scripts ↔ paper objectives

| Script | Papers | Goal |
|--------|--------|------|
| `01_quantum_amplitude_estimation.py` | Brassard; Rebentrost | QAE vs MC error scaling |
| `02_european_option_monte_carlo.py` | Stamatopoulos | Classical option pricing baseline |
| `03_portfolio_optimization_qubo.py` | Level 2A | Markowitz vs discrete QUBO |
| `04_industry_survey_themes.py` | BoF 2025 | Survey + catalog chart |

## 5. Quick demo — QAE error scaling

In [ ]:
import numpy as np

P = 0.37
shots = np.array([50, 100, 500, 1000, 5000])
mc_err = np.sqrt(P * (1 - P) / shots)
qae_err = np.pi / (2 * shots)

plt.loglog(shots, mc_err, "o-", label="Classical MC ~ 1/sqrt(N)")
plt.loglog(shots, qae_err, "s-", label="Ideal QAE ~ 1/N")
plt.xlabel("samples / queries")
plt.ylabel("error on probability")
plt.legend()
plt.title("Core speedup targeted by Rebentrost & Stamatopoulos papers")
plt.grid(True, which="both", alpha=0.3)